# Unstructured Data Final Group Project

Jenn, Leslieane, Harelle, David, and Aryan


##!!!GROUP: RUN CELL 1 THEN SKIP TO PHASE 3!!!

In [1]:
# ============================================================
# CELL 1: Install & import
# ============================================================
!pip install pandas tqdm requests --quiet

import requests
import pandas as pd
import json
import time
import re
from datetime import datetime
from tqdm.notebook import tqdm
from datetime import datetime, timezone

print("✅ Ready")

✅ Ready


In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================

# Reddit blocks the default Python user agent — this one works fine
HEADERS = {
    "User-Agent": "Mozilla/5.0 (career-graph-research-project; educational use)"
}

# Subreddits to scrape
SUBREDDITS = [
    "cscareerquestions",
    "careerguidance",
    "financialindependence",
    "ITCareerQuestions",
    "learnprogramming"
]

# Sort + time filter combos to collect
# Each combo = up to TARGET_POSTS_PER_COMBO posts
SORT_CONFIGS = [
    {"sort": "top",  "t": "year"},
    {"sort": "top",  "t": "month"},
    {"sort": "hot",  "t": None},
]

TARGET_POSTS_PER_COMBO = 500   # will paginate to get this many
MAX_COMMENTS_PER_POST  = 0    # top comments to fetch per post
SLEEP_BETWEEN_REQUESTS = 5.0   # seconds — keeps Reddit happy
MIN_SCORE              = 10
MIN_COMMENTS           = 5

# Career transition keywords — post must contain at least one
CAREER_KEYWORDS = [
    "switch", "transition", "quit", "leave", "change career",
    "new job", "offer", "should i", "advice", "pivot",
    "burnout", "burn out", "laid off", "fired", "job search",
    "salary", "negotiate", "accept", "reject", "interview"
]

OUTPUT_POSTS    = "reddit_posts.csv"
OUTPUT_COMMENTS = "reddit_comments.csv"
OUTPUT_RAW      = "reddit_raw.json"

print("⚙️  Config set")
print(f"   Subreddits    : {len(SUBREDDITS)}")
print(f"   Sort combos   : {len(SORT_CONFIGS)} per subreddit")
print(f"   Target posts  : ~{TARGET_POSTS_PER_COMBO * len(SORT_CONFIGS) * len(SUBREDDITS)} (before filters)")

⚙️  Config set
   Subreddits    : 5
   Sort combos   : 3 per subreddit
   Target posts  : ~7500 (before filters)


In [ ]:
# ============================================================
# CELL 3: Core request helper
# ============================================================

def safe_get(url: str, params: dict = None, retries: int = 3) -> dict | None:
    """
    GET request with retry logic and polite rate limiting.
    Returns parsed JSON or None on failure.
    """
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=15)

            if resp.status_code == 200:
                return resp.json()

            elif resp.status_code == 429:  # rate limited
                wait = int(resp.headers.get('Retry-After', 10))
                print(f"   ⏳ Rate limited — waiting {wait}s")
                time.sleep(wait)

            elif resp.status_code == 403:
                print(f"   🚫 403 Forbidden on {url} — subreddit may be private")
                return None

            elif resp.status_code == 404:
                print(f"   ❌ 404 Not Found: {url}")
                return None

            else:
                print(f"   ⚠️  HTTP {resp.status_code} on attempt {attempt+1}")
                time.sleep(SLEEP_BETWEEN_REQUESTS * 2)

        except requests.exceptions.RequestException as e:
            print(f"   ⚠️  Request error attempt {attempt+1}: {e}")
            time.sleep(SLEEP_BETWEEN_REQUESTS * 2)

    return None


# Quick connectivity test
test = safe_get("https://www.reddit.com/r/careerguidance/top.json",
                params={"limit": 1, "t": "week"})
if test:
    sample_title = test['data']['children'][0]['data']['title']
    print(f"✅ Connected! Sample post: '{sample_title[:70]}...'")
else:
    print("❌ Connection failed — check your internet / try again")

   🚫 403 Forbidden on https://www.reddit.com/r/careerguidance/top.json — subreddit may be private
❌ Connection failed — check your internet / try again


In [ ]:
# ============================================================
# CELL 4: Helper functions
# ============================================================

def has_career_keyword(text: str) -> bool:
    text_lower = (text or "").lower()
    return any(kw in text_lower for kw in CAREER_KEYWORDS)


def has_update(text: str) -> bool:
    """Detects posts/comments where OP shared what actually happened."""
    if not text:
        return False
    patterns = [
        r'\bupdate\b', r'\bEDIT\b', r'follow[- ]?up',
        r'i (did|quit|left|accepted|rejected|got the|took the)',
        r'turns out', r'ended up', r'happy to report',
        r'wanted to share', r'\bresult\b'
    ]
    return bool(re.search('|'.join(patterns), text, re.IGNORECASE))


def parse_post(child: dict, subreddit: str) -> dict:
    """Extract clean fields from a raw Reddit post JSON child."""
    d = child['data']
    body = d.get('selftext', '') or ''
    if body in ('[deleted]', '[removed]'):
        body = ''
    return {
        'post_id'       : d['id'],
        'subreddit'     : subreddit,
        'title'         : d.get('title', ''),
        'body'          : body,
        'flair'         : d.get('link_flair_text') or '',
        'score'         : d.get('score', 0),
        'upvote_ratio'  : d.get('upvote_ratio', 0),
        'num_comments'  : d.get('num_comments', 0),
        'created_utc'   : datetime.fromtimestamp(d['created_utc'], tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
        'created_date'  : datetime.fromtimestamp(d['created_utc'], tz=timezone.utc).strftime('%Y-%m-%d'),
        'url'           : 'https://reddit.com' + d.get('permalink', ''),
        'is_self'       : d.get('is_self', True),
        'has_update'    : has_update(body),
        'has_keyword'   : has_career_keyword(d.get('title', '') + ' ' + body),
        'is_deleted'    : body == ''
    }


def parse_comment(comment_data: dict, post_id: str, subreddit: str) -> dict | None:
    """Extract clean fields from a raw Reddit comment JSON node."""
    if comment_data.get('kind') != 't1':
        return None
    d = comment_data['data']
    body = d.get('body', '') or ''
    if body in ('[deleted]', '[removed]', ''):
        return None
    depth = d.get('depth', 0)
    if depth > 1:   # skip deeply nested replies
        return None
    return {
        'comment_id'    : d['id'],
        'post_id'       : post_id,
        'subreddit'     : subreddit,
        'body'          : body,
        'score'         : d.get('score', 0),
        'depth'         : depth,
        'parent_id'     : d.get('parent_id', ''),
        'author'        : d.get('author', '[deleted]'),
        'created_utc'   : datetime.utcfromtimestamp(d['created_utc']).strftime('%Y-%m-%d %H:%M:%S'),
        'has_update'    : has_update(body)
    }


print("✅ Helper functions ready")

✅ Helper functions ready


In [ ]:
# ============================================================
# CELL 5: Post paginator
# ============================================================

def scrape_subreddit_posts(subreddit: str, sort: str, t: str = None,
                            target: int = 300) -> list[dict]:
    """
    Paginate through a subreddit using Reddit's 'after' cursor.
    Returns a list of raw post dicts.

    Reddit serves max 100 posts per request, so we loop
    and carry the 'after' token forward each time.
    """
    base_url = f"https://www.reddit.com/r/{subreddit}/{sort}.json"
    posts    = []
    after    = None   # pagination cursor — None means start from beginning
    seen_ids = set()

    with tqdm(total=target, desc=f"  r/{subreddit}/{sort}", leave=False) as pbar:
        while len(posts) < target:

            params = {"limit": 100}        # 100 = Reddit's max per page
            if t:                           # time filter (year/month/all)
                params["t"] = t
            if after:                       # carry pagination cursor
                params["after"] = after

            data = safe_get(base_url, params=params)
            if not data:
                break

            children = data.get('data', {}).get('children', [])
            if not children:
                break   # no more posts

            new_this_page = 0
            for child in children:
                if child.get('kind') != 't3':   # t3 = post (t1=comment, t2=account)
                    continue

                post = parse_post(child, subreddit)

                # Dedup
                if post['post_id'] in seen_ids:
                    continue
                seen_ids.add(post['post_id'])

                # Quality filters
                if not post['is_self']:          # skip link posts
                    continue
                if post['score'] < MIN_SCORE:
                    continue
                if post['num_comments'] < MIN_COMMENTS:
                    continue

                posts.append(post)
                new_this_page += 1
                pbar.update(1)

            # Get the cursor for next page
            after = data.get('data', {}).get('after')

            # Reddit returns None for 'after' when there are no more pages
            if not after:
                break

            # Be polite
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    return posts


print("✅ Paginator ready")

✅ Paginator ready


In [ ]:
# ============================================================
# CELL 6: Comment scraper
# ============================================================

def scrape_post_comments(post_id: str, subreddit: str,
                          max_comments: int = 25) -> list[dict]:
    """
    Fetch comments for a single post using the post's JSON endpoint.
    URL format: https://www.reddit.com/r/{sub}/comments/{post_id}.json

    The response is a 2-element list:
      [0] = post data
      [1] = comment tree
    """
    url  = f"https://www.reddit.com/r/{subreddit}/comments/{post_id}.json"
    data = safe_get(url, params={"limit": max_comments, "depth": 2, "sort": "top"})

    if not data or not isinstance(data, list) or len(data) < 2:
        return []

    comment_listing = data[1].get('data', {}).get('children', [])
    comments = []

    for item in comment_listing:
        parsed = parse_comment(item, post_id, subreddit)
        if parsed:
            comments.append(parsed)

        # Also grab one level of replies
        replies = item.get('data', {}).get('replies', '')
        if isinstance(replies, dict):
            for reply in replies.get('data', {}).get('children', []):
                parsed_reply = parse_comment(reply, post_id, subreddit)
                if parsed_reply:
                    comments.append(parsed_reply)

        if len(comments) >= max_comments:
            break

    return comments


print("✅ Comment scraper ready")

✅ Comment scraper ready


In [ ]:
# ============================================================
# CELL 7: Main scraping loop — POSTS ONLY, no comments
# ============================================================

all_posts = []
seen_post_ids = set()

for subreddit in SUBREDDITS:
    print(f"\n{'='*55}")
    print(f"📂 r/{subreddit}")
    print(f"{'='*55}")

    for cfg in SORT_CONFIGS:
        sort = cfg['sort']
        t    = cfg.get('t')
        label = f"{sort}/{t}" if t else sort
        print(f"\n  ↳ {label}")

        posts = scrape_subreddit_posts(
            subreddit, sort, t,
            target=TARGET_POSTS_PER_COMBO
        )

        new_posts = [p for p in posts if p['post_id'] not in seen_post_ids]
        for p in new_posts:
            seen_post_ids.add(p['post_id'])

        all_posts.extend(new_posts)
        print(f"     ✅ Posts: {len(new_posts)} new")

print(f"\n{'='*55}")
print(f"🎉 Done! Total posts: {len(all_posts)}")


📂 r/cscareerquestions

  ↳ top/year


  r/cscareerquestions/top:   0%|          | 0/500 [00:00<?, ?it/s]

   🚫 403 Forbidden on https://www.reddit.com/r/cscareerquestions/top.json — subreddit may be private
     ✅ Posts: 0 new

  ↳ top/month


  r/cscareerquestions/top:   0%|          | 0/500 [00:00<?, ?it/s]

   🚫 403 Forbidden on https://www.reddit.com/r/cscareerquestions/top.json — subreddit may be private
     ✅ Posts: 0 new

  ↳ hot


...

   🚫 403 Forbidden on https://www.reddit.com/r/learnprogramming/top.json — subreddit may be private
     ✅ Posts: 0 new

  ↳ top/month


  r/learnprogramming/top:   0%|          | 0/500 [00:00<?, ?it/s]

   🚫 403 Forbidden on https://www.reddit.com/r/learnprogramming/top.json — subreddit may be private
     ✅ Posts: 0 new

  ↳ hot


  r/learnprogramming/hot:   0%|          | 0/500 [00:00<?, ?it/s]

   🚫 403 Forbidden on https://www.reddit.com/r/learnprogramming/hot.json — subreddit may be private
     ✅ Posts: 0 new

🎉 Done! Total posts: 0


In [ ]:
# ============================================================
# CELL 8: DataFrames + filter to career-relevant posts
# ============================================================

# Define expected columns based on the parse_post function
expected_post_columns = [
    'post_id', 'subreddit', 'title', 'body', 'flair', 'score', 'upvote_ratio',
    'num_comments', 'created_utc', 'created_date', 'url', 'is_self',
    'has_update', 'has_keyword', 'is_deleted'
]

# Initialize df_posts with expected columns even if all_posts is empty
df_posts = pd.DataFrame(all_posts, columns=expected_post_columns)
df_comments = pd.DataFrame()

# Filter posts to career-relevant ones
df_posts_filtered = df_posts[df_posts['has_keyword'] == True].copy()

# Filter comments to match kept posts
kept_ids = set(df_posts_filtered['post_id'])
df_comments_filtered = pd.DataFrame()

df_with_outcome = df_posts_filtered[df_posts_filtered['has_update'] == True]

print("📊 Final dataset:")
print(f"   Posts (all)          : {len(df_posts)}")
print(f"   Posts (career filter): {len(df_posts_filtered)}")
print(f"   Posts (with outcome) : {len(df_with_outcome)}")
print(f"   Comments             : {len(df_comments_filtered)}")
print()
print("Posts per subreddit:")
# Ensure df_posts_filtered is not empty before attempting groupby
if not df_posts_filtered.empty:
    print(df_posts_filtered.groupby('subreddit').size().sort_values(ascending=False))
else:
    print("No filtered posts to display.")
print()
print("Sample top posts:")
# Ensure df_posts_filtered is not empty before attempting sort_values
if not df_posts_filtered.empty:
    print(df_posts_filtered.sort_values('score', ascending=False)[
        ['subreddit','title','score','num_comments','has_update']
    ].head(10))
else:
    print("No filtered posts to display.")


KeyError: 'has_keyword'

In [ ]:
# ============================================================
# CELL 9: EDA
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Reddit Career Advice — Dataset Overview', fontsize=14, fontweight='bold')

# Posts per subreddit
df_posts_filtered.groupby('subreddit').size().sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title('Posts per Subreddit')
axes[0].set_xlabel('Count')

# Score distribution
df_posts_filtered['score'].clip(upper=5000).hist(
    bins=40, ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Post Score Distribution')
axes[1].set_xlabel('Score')

# Posts over time
df_posts_filtered['year_month'] = pd.to_datetime(
    df_posts_filtered['created_date']
).dt.to_period('M')
df_posts_filtered.groupby('year_month').size().plot(ax=axes[2], color='green')
axes[2].set_title('Posts Over Time')
axes[2].set_xlabel('Month')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Saved eda_overview.png")

In [ ]:
# ============================================================
# CELL 10: Save outputs
# ============================================================

df_posts_filtered.to_csv(OUTPUT_POSTS, index=False)
df_comments_filtered.to_csv(OUTPUT_COMMENTS, index=False)

raw_data = {
    "scraped_at"      : datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
    "subreddits"      : SUBREDDITS,
    "total_posts"     : len(df_posts_filtered),
    "total_comments"  : len(df_comments_filtered),
    "posts"           : df_posts_filtered.to_dict(orient='records'),
    "comments"        : df_comments_filtered.to_dict(orient='records')
}
with open(OUTPUT_RAW, 'w') as f:
    json.dump(raw_data, f, indent=2, default=str)

print("💾 Saved:")
print(f"   {OUTPUT_POSTS}    ({len(df_posts_filtered)} rows)")
print(f"   {OUTPUT_COMMENTS} ({len(df_comments_filtered)} rows)")
print(f"   {OUTPUT_RAW}")

try:
    from google.colab import files
    files.download(OUTPUT_POSTS)
    files.download(OUTPUT_COMMENTS)
    files.download(OUTPUT_RAW)
    print("\n📥 Downloads triggered!")
except ImportError:
    print("\n(Not in Colab — files saved locally)")

In [ ]:
# ============================================================
# CELL 11: Preview outcome posts (your graph's gold standard)
# ============================================================

print("=" * 60)
print("POSTS WITH OUTCOME UPDATES — graph gold standard")
print("=" * 60)

sample = df_with_outcome.sample(min(5, len(df_with_outcome)))

for _, row in sample.iterrows():
    print(f"\n📌 [{row['subreddit']}] {row['title']}")
    preview = (row['body'] or '')[:300].replace('\n', ' ')
    print(f"   {preview}...")
    print(f"   Score: {row['score']} | Comments: {row['num_comments']} | {row['created_date']}")
    print(f"   🔗 {row['url']}")

In [ ]:
print("Posts with update:", df_posts_filtered['has_update'].sum())
print("Total filtered posts:", len(df_posts_filtered))

# Phase 2 — LLM Extraction Pipeline
### Reddit Career Advice → Structured JSON → Knowledge Graph

This notebook takes the raw Reddit posts from Phase 1 and uses OpenAI to extract
structured fields from each post — turning messy text into clean graph-ready data.

**Input:** `reddit_posts.csv`  
**Output:** `extracted_posts.json` (structured fields per post)  

**What we extract per post:**
```json
{
  "problem_type": "career switch",
  "current_role": "data analyst",
  "target_role": "software engineer",
  "years_experience": 3,
  "constraints": ["no CS degree", "2 kids", "can't relocate"],
  "advice_summary": "Build projects, do leetcode, apply broadly",
  "outcome": "Got SWE job at mid-size company after 4 months",
  "outcome_sentiment": "positive",
  "has_outcome": true
}
```

---
## Setup
**Before running:** Upload your `reddit_posts.csv` to this Colab session using the file upload button in the left sidebar (folder icon).

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install openai pandas tqdm --quiet
print("✅ Dependencies installed")

✅ Dependencies installed


In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os
import json
import time
import pandas as pd
from openai import OpenAI
from tqdm.notebook import tqdm

print("✅ Imports ready")

✅ Imports ready


In [ ]:
# ============================================================
# CELL 3: 🔑 OpenAI API Key
# ============================================================
OPENAI_API_KEY = "Enter Your API HERE"

client = OpenAI(api_key=OPENAI_API_KEY)

# Quick test
test = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "say ok"}],
    max_tokens=5
)
print(f"✅ OpenAI connected! Response: {test.choices[0].message.content}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: Enter Yo*******HERE. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
# ============================================================
# CELL 4: Configuration
# ============================================================

# Model to use — gpt-4o-mini is cheap (~$0.002 per post) and very good
MODEL = "gpt-4o-mini"

# Input file — upload reddit_posts.csv to Colab first
INPUT_FILE  = "reddit_posts.csv"
OUTPUT_FILE = "extracted_posts.json"
CHECKPOINT  = "checkpoint.json"  # saves progress so you can resume if interrupted

# How many posts to process
# Start with 50 to test, then set to None to process all
MAX_POSTS = None   # None = process everything

# Only process posts with outcomes first (highest value)
OUTCOME_POSTS_ONLY = False  # set True to only process has_update=True posts

# Sleep between API calls to avoid rate limits
SLEEP_BETWEEN_CALLS = 0.5   # seconds

print("⚙️ Config set")
print(f"   Model         : {MODEL}")
print(f"   Max posts     : {MAX_POSTS or 'all'}")
print(f"   Outcome only  : {OUTCOME_POSTS_ONLY}")

In [ ]:
# ============================================================
# CELL 5: Load data
# ============================================================

df = pd.read_csv(INPUT_FILE)
print(f"✅ Loaded {len(df)} posts from {INPUT_FILE}")
print(f"   Columns: {list(df.columns)}")
print(f"   Posts with update flag: {df['has_update'].sum()}")
print()

# Apply filters
if OUTCOME_POSTS_ONLY:
    df = df[df['has_update'] == True].copy()
    print(f"   After outcome filter: {len(df)} posts")

if MAX_POSTS:
    df = df.head(MAX_POSTS)
    print(f"   After max_posts cap: {len(df)} posts")

# Estimate cost
avg_tokens_per_post = 400   # rough estimate
cost_per_1k_tokens  = 0.00015  # gpt-4o-mini input price
est_cost = (len(df) * avg_tokens_per_post / 1000) * cost_per_1k_tokens
print(f"\n💰 Estimated cost: ~${est_cost:.2f} for {len(df)} posts")
print(f"   (gpt-4o-mini is very cheap — this is a rough upper bound)")

In [ ]:
# ============================================================
# CELL 6: Extraction prompt
# ============================================================

SYSTEM_PROMPT = """
You are a career data extraction assistant. Given a Reddit post about a career situation,
extract structured information and return ONLY valid JSON with no extra text, no markdown,
no backticks.

Extract these fields:
- problem_type: one of ["career_switch", "job_search", "salary_negotiation", "layoff",
  "workplace_conflict", "education_decision", "burnout", "promotion", "other"]
- current_role: current job title or field (string, null if not mentioned)
- target_role: desired job/field they want to move into (string, null if not mentioned)
- years_experience: number of years of experience (integer, null if not mentioned)
- industry: current industry (string, null if not mentioned)
- constraints: list of challenges/limitations mentioned (list of short strings, max 5)
- advice_summary: 1-2 sentence summary of the main advice given or sought (string)
- outcome: what actually happened if mentioned - look for EDIT/UPDATE/follow-up language (string, null if no outcome)
- outcome_sentiment: one of ["positive", "negative", "mixed", "unknown"]
- has_outcome: true if the post contains a real outcome/result, false otherwise
- salary_mentioned: true if a specific salary/TC is mentioned
- location: country or city if mentioned (string, null if not mentioned)

Return ONLY the JSON object. No explanation. No markdown.
"""

def build_user_prompt(row) -> str:
    """Build the prompt for a single post."""
    title = row.get('title', '') or ''
    body  = (row.get('body', '') or '')[:2000]  # cap at 2000 chars to save tokens
    sub   = row.get('subreddit', '')
    return f"""Subreddit: r/{sub}
Title: {title}
Post body: {body}"""


def extract_post(row) -> dict:
    """
    Send a single post to the LLM and return structured extraction.
    Returns a dict with extraction fields + original post metadata.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": build_user_prompt(row)}
            ],
            max_tokens=400,
            temperature=0.1   # low temp = consistent structured output
        )

        raw = response.choices[0].message.content.strip()

        # Strip markdown backticks if LLM adds them despite instructions
        raw = raw.replace('```json', '').replace('```', '').strip()

        extracted = json.loads(raw)

        # Attach original metadata
        extracted['post_id']    = row.get('post_id')
        extracted['subreddit']  = row.get('subreddit')
        extracted['title']      = row.get('title')
        extracted['score']      = row.get('score')
        extracted['url']        = row.get('url')
        extracted['created_date'] = row.get('created_date')
        extracted['has_update'] = row.get('has_update')
        extracted['_error']     = None

        return extracted

    except json.JSONDecodeError as e:
        return {
            'post_id'  : row.get('post_id'),
            'title'    : row.get('title'),
            '_error'   : f"JSON parse error: {e}",
            '_raw'     : raw if 'raw' in dir() else None
        }
    except Exception as e:
        return {
            'post_id'  : row.get('post_id'),
            'title'    : row.get('title'),
            '_error'   : str(e)
        }


print("✅ Extraction functions ready")
print()

# Test on one post
print("🧪 Testing on first post...")
test_row = df.iloc[0]
test_result = extract_post(test_row)
import numpy as np

def convert(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)): return bool(obj)
    raise TypeError(f"Not serializable: {type(obj)}")

print(json.dumps(test_result, indent=2, default=convert))

In [ ]:
# ============================================================
# CELL 7: Main extraction loop with checkpointing
# ============================================================
# Checkpointing means if this gets interrupted, you can re-run
# and it will skip already-processed posts

# Load existing checkpoint if it exists
results = []
processed_ids = set()

if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, 'r') as f:
        results = json.load(f)
    processed_ids = {r['post_id'] for r in results if r.get('post_id')}
    print(f"♻️  Resuming from checkpoint: {len(results)} already processed")
else:
    print("🆕 Starting fresh extraction")

# Filter to unprocessed posts
remaining = df[~df['post_id'].isin(processed_ids)]
print(f"   Remaining to process: {len(remaining)} posts")
print()

errors = 0
save_every = 50   # save checkpoint every N posts

for i, (_, row) in enumerate(tqdm(remaining.iterrows(),
                                    total=len(remaining),
                                    desc="Extracting")):
    result = extract_post(row)
    results.append(result)

    if result.get('_error'):
        errors += 1

    # Save checkpoint periodically
    if (i + 1) % save_every == 0:
        with open(CHECKPOINT, 'w') as f:
            json.dump(results, f, default=convert)
        tqdm.write(f"   💾 Checkpoint saved ({len(results)} processed, {errors} errors)")

    time.sleep(SLEEP_BETWEEN_CALLS)

# Final save
with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n🎉 Extraction complete!")
print(f"   Total processed : {len(results)}")
print(f"   Errors          : {errors}")
print(f"   Success rate    : {((len(results)-errors)/len(results)*100):.1f}%")

In [ ]:
# ============================================================
# CELL 8: Analyze extraction results
# ============================================================

df_extracted = pd.DataFrame(results)

# Filter out errors
df_clean = df_extracted[df_extracted['_error'].isna()].copy()
print(f"Clean extractions: {len(df_clean)} / {len(df_extracted)}")
print()

# Problem type distribution
print("📊 Problem Types:")
print(df_clean['problem_type'].value_counts())
print()

# Outcome sentiment
print("📊 Outcome Sentiment:")
print(df_clean['outcome_sentiment'].value_counts())
print()

# Posts with real outcomes
print(f"📊 Posts with real outcomes: {df_clean['has_outcome'].sum()}")
print(f"📊 Posts mentioning salary:  {df_clean['salary_mentioned'].sum()}")
print()

# Top constraints mentioned
print("📊 Sample constraints:")
all_constraints = [c for sublist in df_clean['constraints'].dropna() for c in (sublist if isinstance(sublist, list) else [])]
from collections import Counter
print(Counter(all_constraints).most_common(15))

In [ ]:
# ============================================================
# CELL 9: EDA visualizations
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LLM Extraction Results — Career Advice Dataset', fontsize=14, fontweight='bold')

# Problem types
df_clean['problem_type'].value_counts().plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title('Problem Types')
axes[0].set_xlabel('Count')

# Outcome sentiment
colors = {'positive': 'green', 'negative': 'red', 'mixed': 'orange', 'unknown': 'gray'}
sentiment_counts = df_clean['outcome_sentiment'].value_counts()
sentiment_counts.plot(
    kind='bar',
    ax=axes[1],
    color=[colors.get(x, 'gray') for x in sentiment_counts.index]
)
axes[1].set_title('Outcome Sentiment')
axes[1].set_xlabel('Sentiment')
axes[1].tick_params(axis='x', rotation=45)

# Has outcome vs not
outcome_counts = df_clean['has_outcome'].value_counts()
axes[2].pie(
    outcome_counts,
    labels=['No Outcome', 'Has Outcome'],
    colors=['#ccc', 'steelblue'],
    autopct='%1.0f%%',
    startangle=90
)
axes[2].set_title('Posts With Real Outcomes')

plt.tight_layout()
plt.savefig('extraction_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Saved extraction_eda.png")

In [ ]:
# ============================================================
# CELL 10: Save final outputs
# ============================================================

# Save clean JSON for knowledge graph pipeline
clean_records = df_clean.to_dict(orient='records')
with open('extracted_posts_clean.json', 'w') as f:
    json.dump(clean_records, f, indent=2, default=convert)

# Save as CSV too
df_clean.to_csv('extracted_posts_clean.csv', index=False)

# Save outcome posts separately — these are your graph gold standard
df_outcomes = df_clean[df_clean['has_outcome'] == True]
df_outcomes.to_csv('outcome_posts.csv', index=False)

print("💾 Saved:")
print(f"   extracted_posts_clean.json  ({len(df_clean)} records)")
print(f"   extracted_posts_clean.csv   ({len(df_clean)} rows)")
print(f"   outcome_posts.csv           ({len(df_outcomes)} rows)")

# Download from Colab
try:
    from google.colab import files
    files.download('extracted_posts_clean.json')
    files.download('extracted_posts_clean.csv')
    files.download('outcome_posts.csv')
    print("\n📥 Downloads triggered!")
except ImportError:
    print("\n(Not in Colab — files saved locally)")

In [ ]:
# Run this in a new cell
sample = df_clean[df_clean['has_outcome'] == True].sample(3)
for _, row in sample.iterrows():
    print(f"\n📌 {row['title'][:70]}")
    print(f"   Type       : {row['problem_type']}")
    print(f"   From → To  : {row['current_role']} → {row['target_role']}")
    print(f"   Constraints: {row['constraints']}")
    print(f"   Outcome    : {row['outcome']}")
    print(f"   Sentiment  : {row['outcome_sentiment']}")

# Phase 3 — Knowledge Graph


In [2]:
# ============================================================
# PHASE 3 — CELL 1: Install & import
# ============================================================
!pip install networkx openai scikit-learn numpy --quiet

import json
import os
import time
import pickle
import numpy as np
import networkx as nx
from collections import Counter
from openai import OpenAI

print("✅ Phase 3 ready")

✅ Phase 3 ready


In [3]:
# ============================================================
# PHASE 3 — CELL 2: Config
# ===========================================================

from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

INPUT_FILE  = "extracted_posts_clean.json"   # from Phase 2
GRAPH_FILE  = "career_graph.gpickle"
STATS_FILE  = "graph_stats.json"
EMBED_CACHE = "embeddings_cache.json"

SIM_THRESHOLD = 0.6 # TEMPORARILTY REDUCING - jenn
#SIM_THRESHOLD = 0.82
MAX_SIM_EDGES = 5
EMBED_MODEL   = "text-embedding-3-small"

print(f"⚙️  Config set")
print(f"   Input         : {INPUT_FILE}")
print(f"   Sim threshold : {SIM_THRESHOLD}")
print(f"   Max sim edges : {MAX_SIM_EDGES}")

⚙️  Config set
   Input         : extracted_posts_clean.json
   Sim threshold : 0.6
   Max sim edges : 5


In [4]:
# Upload extracted_posts_clean.json to Colab
from google.colab import files
uploaded = files.upload()   # pick extracted_posts_clean.json from your computer

Saving extracted_posts_clean.json to extracted_posts_clean (1).json


In [5]:
# ============================================================
# PHASE 3 — CELL 3: Load Phase 2 extracted data
# ============================================================

with open(INPUT_FILE, 'r') as f:
    records = json.load(f)

# Drop any records that errored during Phase 2 extraction
records = [r for r in records if r.get('post_id') and not r.get('_error')]

print(f"✅ Loaded {len(records)} clean records")

sample = records[0]
print(f"\nSample record:")
for k in ['post_id','problem_type','current_role','target_role','constraints','has_outcome','outcome_sentiment']:
    print(f"   {k:<20} : {sample.get(k)}")

✅ Loaded 1897 clean records

Sample record:
   post_id              : 1mcop22
   problem_type         : career_switch
   current_role         : IT developer
   target_role          : sales
   constraints          : ['rejections from companies', 'ghost jobs', 'interned for free', 'grinding Leetcode', 'negative comments']
   has_outcome          : True
   outcome_sentiment    : positive


In [6]:
# ============================================================
# PHASE 3 — CELL 4: Build knowledge graph
# ============================================================

G = nx.DiGraph()

def add_node_once(graph, node_id, **attrs):
    """Add a node only if it doesn't already exist (MERGE behavior)."""
    if not graph.has_node(node_id):
        graph.add_node(node_id, **attrs)

def norm(s):
    """Normalize a string into a clean node ID."""
    if not s:
        return None
    return str(s).strip().lower().replace(' ', '_').replace('/', '_')[:80]

skipped = 0
for rec in records:
    pid = rec.get('post_id')
    if not pid:
        skipped += 1
        continue

    # ── CareerSituation (one per post) ──────────────────────────────────────
    sit_id = f"situation:{pid}"
    add_node_once(G, sit_id,
        node_type      = "CareerSituation",
        post_id        = pid,
        title          = rec.get('title', ''),
        subreddit      = rec.get('subreddit', ''),
        score          = rec.get('score', 0),
        url            = rec.get('url', ''),
        created_date   = rec.get('created_date', ''),
        advice_summary = rec.get('advice_summary', ''),
        has_outcome    = bool(rec.get('has_outcome')),
    )

    # ── ProblemType ─────────────────────────────────────────────────────────
    pt = norm(rec.get('problem_type'))
    if pt:
        pt_id = f"problem:{pt}"
        add_node_once(G, pt_id, node_type="ProblemType", label=pt)
        G.add_edge(sit_id, pt_id, edge_type="HAS_PROBLEM")

    # ── Role: current ───────────────────────────────────────────────────────
    curr = norm(rec.get('current_role'))
    if curr:
        r_id = f"role:{curr}"
        add_node_once(G, r_id, node_type="Role", label=curr)
        G.add_edge(sit_id, r_id, edge_type="CURRENTLY_IN")

    # ── Role: target ────────────────────────────────────────────────────────
    tgt = norm(rec.get('target_role'))
    if tgt:
        r_id = f"role:{tgt}"
        add_node_once(G, r_id, node_type="Role", label=tgt)
        G.add_edge(sit_id, r_id, edge_type="TARGETING")

    # ── Constraints ─────────────────────────────────────────────────────────
    constraints = rec.get('constraints') or []
    if isinstance(constraints, str):
        constraints = [c.strip() for c in constraints.split(',')]
    for c in constraints[:5]:
        c_norm = norm(c)
        if c_norm:
            c_id = f"constraint:{c_norm}"
            add_node_once(G, c_id, node_type="Constraint", label=c_norm)
            G.add_edge(sit_id, c_id, edge_type="FACES_CONSTRAINT")

    # ── Advice ──────────────────────────────────────────────────────────────
    advice = (rec.get('advice_summary') or '').strip()
    if advice:
        adv_id = f"advice:{pid}"
        add_node_once(G, adv_id, node_type="Advice", text=advice)
        G.add_edge(sit_id, adv_id, edge_type="HAS_ADVICE")

    # ── Outcome ─────────────────────────────────────────────────────────────
    if rec.get('has_outcome') and rec.get('outcome'):
        out_id = f"outcome:{pid}"
        add_node_once(G, out_id,
            node_type = "Outcome",
            text      = rec['outcome'],
            sentiment = rec.get('outcome_sentiment', 'unknown'),
        )
        G.add_edge(sit_id, out_id,
            edge_type = "LEADS_TO",
            sentiment = rec.get('outcome_sentiment', 'unknown')
        )

print(f"✅ Graph built")
print(f"   Nodes : {G.number_of_nodes():,}")
print(f"   Edges : {G.number_of_edges():,}")
print(f"   Skipped : {skipped}")

type_counts = Counter(data['node_type'] for _, data in G.nodes(data=True))
print(f"\nNode type breakdown:")
for ntype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"   {ntype:<20} : {count:,}")

✅ Graph built
   Nodes : 10,946
   Edges : 12,421
   Skipped : 0

Node type breakdown:
   Constraint           : 5,684
   CareerSituation      : 1,897
   Advice               : 1,897
   Role                 : 991
   Outcome              : 467
   ProblemType          : 10


PLEASE DO NOT RERUN THE CELL BELOW!!

In [ ]:
# ============================================================
# PHASE 3 — CELL 5: Embed situations + add SIMILAR_SITUATION edges
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

# Load cache
if os.path.exists(EMBED_CACHE):
    with open(EMBED_CACHE, 'r') as f:
        embed_cache = json.load(f)
    print(f"📂 Loaded {len(embed_cache)} cached embeddings")
else:
    embed_cache = {}
    print("📂 No cache — embedding from scratch")

sit_nodes = [
    (nid, data) for nid, data in G.nodes(data=True)
    if data.get('node_type') == 'CareerSituation'
]
print(f"   Situation nodes to embed: {len(sit_nodes)}")

def build_embed_text(nid, data):
    return f"{data.get('title', '')}. {data.get('advice_summary', '')}".strip()

def embed_batch(ids_texts):
    """Embed (id, text) pairs not already in cache. Updates embed_cache in-place."""
    to_embed = [(i, t) for i, t in ids_texts if i not in embed_cache]
    if not to_embed:
        print("   All already cached!")
        return
    BATCH = 100
    for start in range(0, len(to_embed), BATCH):
        batch = to_embed[start:start+BATCH]
        b_ids, b_texts = zip(*batch)
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=list(b_texts))
            for idx, emb_obj in enumerate(resp.data):
                embed_cache[b_ids[idx]] = emb_obj.embedding
            print(f"   ✅ Batch {start//BATCH + 1} done ({len(batch)} items)")
        except Exception as e:
            print(f"   ⚠️  Batch failed: {e}")
        time.sleep(0.3)

ids_texts = [(nid, build_embed_text(nid, data)) for nid, data in sit_nodes]
embed_batch(ids_texts)

with open(EMBED_CACHE, 'w') as f:
    json.dump(embed_cache, f)
print(f"\n💾 Cache saved ({len(embed_cache)} embeddings)")

# Build similarity matrix
valid     = [(nid, embed_cache[nid]) for nid, _ in sit_nodes if nid in embed_cache]
valid_ids = [v[0] for v in valid]
valid_vecs = np.array([v[1] for v in valid])

print(f"\n📐 Computing similarity matrix ({len(valid_ids)} × {len(valid_ids)})...")
sim_matrix = cosine_similarity(valid_vecs)
print("   Done!")

# Add edges
sim_edge_count = 0
for i, nid_i in enumerate(valid_ids):
    sims  = sim_matrix[i]
    top_j = np.argsort(sims)[::-1]
    added = 0
    for j in top_j:
        if i == j:
            continue
        score = float(sims[j])
        if score < SIM_THRESHOLD or added >= MAX_SIM_EDGES:
            break
        nid_j = valid_ids[j]
        if not G.has_edge(nid_i, nid_j):
            G.add_edge(nid_i, nid_j,
                edge_type  = "SIMILAR_SITUATION",
                similarity = round(score, 4)
            )
            sim_edge_count += 1
            added += 1

print(f"\n✅ Added {sim_edge_count:,} SIMILAR_SITUATION edges (threshold={SIM_THRESHOLD})")
print(f"   Total edges now: {G.number_of_edges():,}")

In [7]:
# ============================================================
# PHASE 3 — CELL 6: Centrality analysis
# ===========================================================

import warnings
warnings.filterwarnings('ignore')

in_deg = dict(G.in_degree())

def top_nodes_by_type(node_type, top_n=10):
    nodes = [
        (nid, data, in_deg.get(nid, 0))
        for nid, data in G.nodes(data=True)
        if data.get('node_type') == node_type
    ]
    nodes.sort(key=lambda x: -x[2])
    return nodes[:top_n]

print("=" * 60)
print("CENTRALITY ANALYSIS")
print("=" * 60)

print("\nTop ProblemTypes (by in-degree):")
for nid, data, deg in top_nodes_by_type("ProblemType"):
    print(f"   {data.get('label', nid):<35}  in-degree: {deg}")

print("\nTop Roles (by in-degree):")
for nid, data, deg in top_nodes_by_type("Role"):
    print(f"   {data.get('label', nid):<35}  in-degree: {deg}")

print("\nTop Constraints (by in-degree):")
for nid, data, deg in top_nodes_by_type("Constraint"):
    print(f"   {data.get('label', nid):<35}  in-degree: {deg}")

print("\nOutcome Sentiment Breakdown:")
sentiments = [
    data.get('sentiment', 'unknown')
    for _, data in G.nodes(data=True)
    if data.get('node_type') == 'Outcome'
]
sent_counts = Counter(sentiments)
total = sum(sent_counts.values())
for s, cnt in sorted(sent_counts.items(), key=lambda x: -x[1]):
    print(f"   {s:<12}  {cnt:>4}  ({cnt/total*100:.1f}%)")

print("\nMost-connected CareerSituations (SIMILAR_SITUATION out-degree):")
sim_deg = [
    (nid, data, sum(1 for _, _, ed in G.out_edges(nid, data=True)
                    if ed.get('edge_type') == 'SIMILAR_SITUATION'))
    for nid, data in G.nodes(data=True)
    if data.get('node_type') == 'CareerSituation'
]
sim_deg.sort(key=lambda x: -x[2])
for nid, data, deg in sim_deg[:10]:
    print(f"   [{deg} neighbors] {data.get('title','')[:65]}")

CENTRALITY ANALYSIS

Top ProblemTypes (by in-degree):
   job_search                           in-degree: 462
   other                                in-degree: 326
   career_switch                        in-degree: 288
   education_decision                   in-degree: 234
   burnout                              in-degree: 183
   workplace_conflict                   in-degree: 168
   layoff                               in-degree: 122
   salary_negotiation                   in-degree: 81
   promotion                            in-degree: 32
   retirement_decision                  in-degree: 1

Top Roles (by in-degree):
   software_engineer                    in-degree: 69
   it                                   in-degree: 62
   help_desk                            in-degree: 33
   programmer                           in-degree: 27
   developer                            in-degree: 25
   software_developer                   in-degree: 24
   engineer                             in-degree

In [9]:
# ============================================================
# PHASE 3 — CELL 7: Save graph + stats
# ============================================================

with open(GRAPH_FILE, 'wb') as f:
    pickle.dump(G, f)
print(f" {GRAPH_FILE}")

type_counts      = Counter(data['node_type'] for _, data in G.nodes(data=True))
edge_type_counts = Counter(ed.get('edge_type', 'unknown') for _, _, ed in G.edges(data=True))
sentiment_counts = Counter(
    data.get('sentiment', 'unknown')
    for _, data in G.nodes(data=True)
    if data.get('node_type') == 'Outcome'
)

stats = {
    "total_nodes"       : G.number_of_nodes(),
    "total_edges"       : G.number_of_edges(),
    "node_types"        : dict(type_counts),
    "edge_types"        : dict(edge_type_counts),
    "outcome_sentiment" : dict(sentiment_counts),
    "top_problem_types" : [
        {"label": data.get('label'), "in_degree": in_deg.get(nid, 0)}
        for nid, data, _ in top_nodes_by_type("ProblemType", top_n=8)
    ],
    "top_roles"         : [
        {"label": data.get('label'), "in_degree": in_deg.get(nid, 0)}
        for nid, data, _ in top_nodes_by_type("Role", top_n=10)
    ],
    "top_constraints"   : [
        {"label": data.get('label'), "in_degree": in_deg.get(nid, 0)}
        for nid, data, _ in top_nodes_by_type("Constraint", top_n=10)
    ],
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)
print(f" {STATS_FILE}")

try:
    from google.colab import files
    files.download(GRAPH_FILE)
    files.download(STATS_FILE)
    files.download(EMBED_CACHE)
    print("\n Downloads triggered!")
except ImportError:
    print("\n(Not in Colab — files saved locally)")

print(f"\nPhase 3 complete! Nodes: {G.number_of_nodes():,} | Edges: {G.number_of_edges():,}")

 career_graph.gpickle
 graph_stats.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Downloads triggered!

Phase 3 complete! Nodes: 10,946 | Edges: 12,421


In [10]:
# ============================================================
# PHASE 3 — CELL 8: Sanity check — inspect one node's neighbors
# ============================================================

best_nid, best_data = max(
    ((nid, data) for nid, data in G.nodes(data=True)
     if data.get('node_type') == 'CareerSituation'),
    key=lambda x: x[1].get('score', 0)
)

print(f"🔍 Inspecting: {best_data.get('title', '')[:70]}")
print(f"   Score    : {best_data.get('score')}")
print(f"   Subreddit: r/{best_data.get('subreddit')}")

print(f"\nOutgoing edges:")
for _, target, edata in G.out_edges(best_nid, data=True):
    tdata = G.nodes[target]
    label = tdata.get('label') or tdata.get('text', '')[:50] or target
    sim   = f"  sim={edata['similarity']}" if 'similarity' in edata else ''
    print(f"   [{edata.get('edge_type'):<22}] → {label}{sim}")

print(f"\nIncoming SIMILAR_SITUATION:")
similar_in = [
    (src, G.nodes[src], edata.get('similarity', 0))
    for src, _, edata in G.in_edges(best_nid, data=True)
    if edata.get('edge_type') == 'SIMILAR_SITUATION'
]
similar_in.sort(key=lambda x: -x[2])
for src, sdata, sim in similar_in[:5]:
    print(f"   [{sim:.3f}] {sdata.get('title','')[:65]}")

print("\nGraph looks correct — ready for Phase 4!")

🔍 Inspecting: I made use of my "Fuck You" money today.
   Score    : 20156
   Subreddit: r/financialindependence

Outgoing edges:
   [HAS_PROBLEM           ] → layoff
   [CURRENTLY_IN          ] → c-level_exec
   [TARGETING             ] → home_depot_employee
   [FACES_CONSTRAINT      ] → bad_vibes_from_ceo
   [FACES_CONSTRAINT      ] → unethical_financial_practices
   [FACES_CONSTRAINT      ] → desire_to_avoid_corporate_office_bullshit
   [HAS_ADVICE            ] → The individual sought advice on leaving a toxic wo
   [LEADS_TO              ] → Resigned from the new job immediately after realiz

Incoming SIMILAR_SITUATION:

Graph looks correct — ready for Phase 4!


## Phase 4 — GraphRAG Q&A Pipeline

In [11]:
from google.colab import userdata
import numpy as np
from openai import OpenAI
import pickle

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

#load graph
with open(GRAPH_FILE, 'rb') as f:
    G = pickle.load(f)

In [12]:
# ============================================================
# PHASE 4 — FIX: Inject embeddings from cache into graph nodes
# ============================================================
import json

# Load the embedding cache saved in Phase 3
with open("embeddings_cache.json", "r") as f:
    embed_cache = json.load(f)

print(f"📂 Loaded {len(embed_cache)} cached embeddings")

# Write them back into the graph nodes
injected = 0
for nid, data in G.nodes(data=True):
    if data.get("node_type") == "CareerSituation" and nid in embed_cache:
        G.nodes[nid]["embedding"] = embed_cache[nid]
        injected += 1

print(f"✅ Injected embeddings into {injected} CareerSituation nodes")

📂 Loaded 1897 cached embeddings
✅ Injected embeddings into 1897 CareerSituation nodes


In [13]:
# Convert user text into a vector embedding
def embed_query(query):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    return np.array(response.data[0].embedding)

In [23]:
# Measure similarity between vectors
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Retrieve most similar reddit posts to situation
# def get_top_k_similar_from_graph(G, query_embedding, k=5):
#     scored = []

#     for node, data in G.nodes(data=True):
#         if data.get("node_type") == "CareerSituation" and "embedding" in data:
#             emb = np.array(data["embedding"])
#             score = cosine_similarity(query_embedding, emb)

#             scored.append({
#                 "id": node,
#                 "text": data.get("text"),
#                 "score": float(score)
#             })

#     return sorted(scored, key=lambda x: x["score"], reverse=True)[:k]

def get_top_k_similar_from_graph(G, query_embedding, k=5, threshold=0.3):
    scored = []

    for node, data in G.nodes(data=True):
        if data.get("node_type") == "CareerSituation" and "embedding" in data:
            emb = np.array(data["embedding"])
            score = cosine_similarity(query_embedding, emb)

            if score >= threshold:
                scored.append({
                    "id": node,
                    "text": data.get("title") or data.get("advice_summary", ""),

                    "score": float(score)
                })

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:k]

In [24]:

# Traverse edges
def get_graph_context(graph, situation_ids):
    context = []
    for sit_id in situation_ids:
        if sit_id not in graph:
            continue

        advice_texts = set()
        outcome_texts = set()
        constraint_labels = set()
        problem_type_labels = set()

        # Iterate over immediate neighbors
        for _, neighbor_id, edge_data in graph.out_edges(sit_id, data=True):
            edge_type = edge_data.get('edge_type')
            neighbor_data = graph.nodes[neighbor_id]
            node_type = neighbor_data.get('node_type')

            if edge_type == "HAS_ADVICE" and node_type == 'Advice':
                advice_texts.add(neighbor_data.get('text'))
            elif edge_type == "LEADS_TO" and node_type == 'Outcome':
                outcome_texts.add(neighbor_data.get('text'))
            elif edge_type == "FACES_CONSTRAINT" and node_type == 'Constraint':
                constraint_labels.add(neighbor_data.get('label'))
            elif edge_type == "HAS_PROBLEM" and node_type == 'ProblemType':
                problem_type_labels.add(neighbor_data.get('label'))

        context.append({
            "id": sit_id,
            "advice": list(advice_texts),
            "outcomes": list(outcome_texts),
            "constraints": list(constraint_labels),
            "problems": list(problem_type_labels),
        })
    return context

In [25]:
## adding in debug
def has_real_context(context):
    return any(
        len(c["advice"]) > 0 or
        len(c["outcomes"]) > 0 or
        len(c["constraints"]) > 0
        for c in context
    )

In [26]:
# Create grounded prompt
def build_prompt(user_query, similar_cases, graph_context):
    return f"""
You are a career advisor using real-world data from Reddit.

USER SITUATION:
{user_query}

SIMILAR PAST SITUATIONS (with similarity scores):
{similar_cases}

GRAPH EVIDENCE (structured):
{graph_context}

INSTRUCTIONS:
- Base your answer ONLY on the provided graph evidence
- Identify common patterns in advice, constraints, and outcomes
- Be specific and practical
- If patterns conflict, explain tradeoffs
- If the graph evidence is weak or minimal, explicitly say that
- DO NOT generate general advice without evidence
- Do NOT hallucinate anything outside the data

OUTPUT:
A clear, structured response explaining:
1. What people in similar situations did
2. What worked vs didn’t
3. Key constraints to consider
4. Actionable advice
"""

In [27]:
# Use GPT to synthesize structured insights
def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a grounded, data-driven career advisor."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

In [19]:
def graph_rag_pipeline(user_query, k=5):
    query_embedding = embed_query(user_query)

    similar = get_top_k_similar_from_graph(G, query_embedding, k)
    ids = [s["id"] for s in similar]

    context = get_graph_context(G, ids)

    # =========================
    # 🔍 DEBUG BLOCK
    # =========================
    print("\n--- DEBUG START ---")

    print("\nTop Similar Cases:")
    for s in similar:
        print(f"Score: {s['score']:.3f} | Text: {str(s['text'])[:80]}")

    print("\nGraph Context:")
    for c in context:
        print(c)

    print("\n--- DEBUG END ---\n")
    # =========================

    # 👇 check if context is meaningful
    if not has_real_context(context):
        return {
            "similar_cases": similar,
            "graph_context": context,
            "answer": "We couldn't find strong matching situations with meaningful data in the dataset."
        }

    prompt = build_prompt(user_query, similar, context)
    answer = generate_answer(prompt)

    return {
        "similar_cases": similar,
        "graph_context": context,
        "answer": answer
    }

In [28]:
query = "I'm looking for a software engineering job"

result = graph_rag_pipeline(query)

print(result["answer"])


--- DEBUG START ---

Top Similar Cases:
Score: 0.576 | Text: "How to level up as a Software Engineering?– seeking advice
Score: 0.547 | Text: What can I pivot to from Software Engineering
Score: 0.545 | Text: Morale is so low in our engineering department since our manager was laid off th
Score: 0.544 | Text: advice for aspiring software engineer!
Score: 0.541 | Text: Anyone else tired of the software field?

Graph Context:
{'id': 'situation:1jz4bnt', 'advice': ['The poster is seeking advice on how to actively improve their technical skills and expertise in software engineering.'], 'outcomes': [], 'constraints': ['not_improving_as_fast_as_hoped', 'want_to_grow_expertise_actively'], 'problems': ['other']}
{'id': 'situation:1kmpzw4', 'advice': ['The poster is seeking advice on potential career pivots from software engineering, particularly considering a transition to a PA role that requires further education.'], 'outcomes': [], 'constraints': ['laid_off', 'hostile_work_environment', 'po

In [29]:
count_total = 0
count_with_embeddings = 0

for _, data in G.nodes(data=True):
    if data.get("node_type") == "CareerSituation":
        count_total += 1
        if "embedding" in data:
            count_with_embeddings += 1

print("Total CareerSituations:", count_total)
print("With embeddings:", count_with_embeddings)

Total CareerSituations: 1897
With embeddings: 1897


In [31]:
query = "I'm wanting to switch from manufacturing to marketing"

result = graph_rag_pipeline(query)

print(result["answer"])


--- DEBUG START ---

Top Similar Cases:
Score: 0.550 | Text: I studied marketing, was pulled into sales coordination, and now I'm only good w
Score: 0.529 | Text: How do I get my spark back?
Score: 0.498 | Text: Burnt Out. Looking to switch fields
Score: 0.474 | Text: Considering switching industries
Score: 0.463 | Text: I’m thinking about leaving software development. With the layoffs and increasing

Graph Context:
{'id': 'situation:1sihdgo', 'advice': ['The poster is seeking guidance on how to transition from a sales coordinator role to a more specialized position in data analysis or systems automation, given their lack of deep knowledge in any area.'], 'outcomes': [], 'constraints': ['managing_older_colleagues', 'salary_not_reflective_of_role', 'fear_of_being_stuck_in_a_loop', 'lack_of_specialization', 'sales_targets_to_meet'], 'problems': ['career_switch']}
{'id': 'situation:1s2sygq', 'advice': ['The poster is seeking recommendations for a job with more structure and consistency, 

### Phase 5 is ready to start!